# Data Loading Demo Using a Custom-Configured Datamodule

This notebook demonstrates how to configure and override default settings for datamodules, and how to fetch and visualize individual data samples from a dataset. It is meant as a simplistic demo of how to actually prepare
raw data for experiments with HuggingFace's TRL.

In [ ]:
# import necessary modules (bare minimum)
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.organisms.datamodules.shortcuts_configs as pyine_dm_configs
import pyine.prompts.types

In [ ]:
# fetch the necessary dataset paths to create a proper datamodule config
dataset_paths = sorted(
    pyine.data.traces.dataset_utils.get_matching_dataset_paths(
        source_dataset_name="TACO",
        pattern="v1.5/10s10t.*of000026.*.lmdb",
    )
)
# let's just keep a single part for a quick demo
dataset_paths = dataset_paths[:1]
print(f"selected dataset paths:\n\t{'\n\t'.join([str(p) for p in dataset_paths])}")
dataset_split_file_path = pyine.data.utils.splits.get_dataset_split_file_path("TACO")
print(f"split file path: {dataset_split_file_path}")

In [ ]:
# set up a new datamodule config with some overrides, e.g. a new prompt config version
datamodule_config = pyine_dm_configs.get_datamodule_config(
    lmdb_paths=dataset_paths,
    split_file_path=dataset_split_file_path,
    seed=0,
    as_pydantic=True,
    # all settings below will override defaults from the base config
    prompt_config=pyine.prompts.types.PromptBuildConfig(
        prompt_name=pyine.prompts.PromptNames.CODE_EXECUTION,
        version="rl_tagged_answer",
        use_chat_template=True,
        include_examples=False,
    ),
    min_samples_hinted=0,  # also, let's make this setting less restrictive/annoying
    hf_messages_key="prompt",  # to adjust depending on the key name expected downstream
)
print(datamodule_config.model_dump_json(indent=2))

In [ ]:
# now, instantiate the actual datamodule and initialize its metadata
dm = datamodule_config.instantiate_datamodule(verbose=True)
dm.prepare_data()
dm.setup()

In [ ]:
# finally, we should be able to get an RL-ready dataset from the datamodule
dataset = dm.get_hf_messages_dataset(
    subset_name="train",
    append_answer=False,  # no answers for RL
    merge_system_with_user=False,  # to adjust depending on whether we want system messages too
    keep_original_data=True,  # needed to compute rewards
)
print(f"sample count: {len(dataset)}")

In [ ]:
# let's take a look at what is inside a single sample (remember: we kept all metadata as an option above)
sample_idx = 0  # pick an arbitrary sample
raw_sample = dataset[sample_idx]
raw_sample_keys = list(raw_sample.keys())
print(f"sample keys={raw_sample_keys}")
print(f"identifier={raw_sample['identifier']}")
sample_messages = {m["role"]: m["content"] for m in raw_sample[datamodule_config.hf_messages_key]}
print(f"sample messages:\n\t{'\n\t'.join([repr(k + ": " + m[:80] + "...") for k, m in sample_messages.items()])}")